# MELD Text Fine-Tuning Pipeline: DistillRoBERTa-> SDT Features

This notebook builds a complete text-model workflow on top of `meld_multimodal_features.pkl`:

1. Fine-tune **DistillRoBERTa** for utterance-level emotion classification.
2. Report validation/test metrics (`classification_report`, confusion matrix, weighted F1).
3. Save the best checkpoint and run sentence-level inference.
4. Regenerate MELD text features from the best checkpoint so `train.py` can still train multimodally from a pickle file.

# Notes Before Running

- This notebook assumes execution from `Self-Distillation_Transformer/`.
- It uses utterance text in the MELD pickle (`videoSentence`) and labels (`videoLabels`).
- Fine-tuning large language models can be GPU intensive.

## dependency install (uncomment if needed)

In [ ]:
# !pip install -q transformers accelerate scikit-learn seaborn tqdm

In [ ]:
import os
import random
import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'meld_multimodal_features.pkl'
CHECKPOINT_ROOT = ROOT / 'notebook_checkpoints'
CHECKPOINT_ROOT.mkdir(exist_ok=True)
print('Data path:', DATA_PATH)

# 1) Load MELD Pickle and Build Utterance-Level Dataset

In [ ]:
with open(DATA_PATH, 'rb') as f:
    meld_data = pickle.load(f)

(
    videoIDs,
    videoSpeakers,
    videoLabels,
    videoText,
    roberta2,
    roberta3,
    roberta4,
    videoAudio,
    videoVisual,
    videoSentence,
    trainVid,
    testVid,
    _,
) = meld_data

print('Num train dialogs:', len(trainVid))
print('Num test dialogs:', len(testVid))
print('Num dialogs in sentence dict:', len(videoSentence))

In [ ]:
# MELD 7-class mapping used by many prior works.
label2name = {
    0: 'neutral',
    1: 'surprise',
    2: 'fear',
    3: 'sadness',
    4: 'joy',
    5: 'disgust',
    6: 'anger',
}
name2label = {v: k for k, v in label2name.items()}
num_labels = len(label2name)


def flatten_dialogs(dialog_ids, split_name):
    rows = []
    for vid in dialog_ids:
        sentences = videoSentence[vid]
        labels = videoLabels[vid]
        for idx, (txt, lab) in enumerate(zip(sentences, labels)):
            rows.append(
                {
                    'dialog_id': vid,
                    'utterance_id': idx,
                    'text': str(txt),
                    'label': int(lab),
                    'split': split_name,
                }
            )
    return pd.DataFrame(rows)

train_df_full = flatten_dialogs(list(trainVid), 'train')
test_df = flatten_dialogs(list(testVid), 'test')

# Hold out part of train split for validation.
train_df, valid_df = train_test_split(
    train_df_full,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df_full['label'],
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('Train size:', len(train_df))
print('Valid size:', len(valid_df))
print('Test size :', len(test_df))
print('Train label distribution:', dict(sorted(Counter(train_df['label']).items())))

# 2) Data + Training Utilities

In [ ]:
class MELDTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = int(self.labels[idx])
        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item


def build_dataloaders(tokenizer, train_df, valid_df, test_df, batch_size=16, max_length=128):
    train_ds = MELDTextDataset(train_df, tokenizer, max_length=max_length)
    valid_ds = MELDTextDataset(valid_df, tokenizer, max_length=max_length)
    test_ds = MELDTextDataset(test_df, tokenizer, max_length=max_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, valid_loader, test_loader


def run_eval(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_preds = [], []

    with torch.no_grad():
        for batch in dataloader:
            labels = batch['labels'].to(device)
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            outputs = model(**inputs)
            logits = outputs.logits
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            preds = torch.argmax(logits, dim=1)

            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())

    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    return {
        'loss': avg_loss,
        'accuracy': acc,
        'f1_weighted': f1_weighted,
        'labels': all_labels,
        'preds': all_preds,
    }


def plot_conf_mat(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(label2name.keys()))
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=[label2name[i] for i in range(num_labels)],
        yticklabels=[label2name[i] for i in range(num_labels)],
    )
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()


def print_eval_report(eval_out, split_name='split'):
    print(f"\n[{split_name}] loss={eval_out['loss']:.4f} | acc={eval_out['accuracy']:.4f} | f1_weighted={eval_out['f1_weighted']:.4f}")
    print(
        classification_report(
            eval_out['labels'],
            eval_out['preds'],
            labels=list(label2name.keys()),
            target_names=[label2name[i] for i in range(num_labels)],
            digits=4,
            zero_division=0,
        )
    )

In [ ]:
def finetune_model(
    pretrained_name,
    run_name,
    train_df,
    valid_df,
    test_df,
    epochs=3,
    batch_size=16,
    lr=2e-5,
    max_length=128,
):
    run_dir = CHECKPOINT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_dir = run_dir / 'best_checkpoint'

    tokenizer = AutoTokenizer.from_pretrained(pretrained_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_name,
        num_labels=num_labels,
    ).to(device)

    train_loader, valid_loader, test_loader = build_dataloaders(
        tokenizer,
        train_df,
        valid_df,
        test_df,
        batch_size=batch_size,
        max_length=max_length,
    )

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(num_labels),
        y=train_df['label'].values,
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_valid_f1 = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        pbar = tqdm(train_loader, desc=f'{run_name} | epoch {epoch}/{epochs}')
        for batch in pbar:
            labels = batch['labels'].to(device)
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}

            optimizer.zero_grad()
            outputs = model(**inputs)
            logits = outputs.logits
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item() * labels.size(0)
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        train_loss = running_loss / len(train_loader.dataset)
        valid_out = run_eval(model, valid_loader, criterion)
        test_out = run_eval(model, test_loader, criterion)

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'valid_loss': valid_out['loss'],
            'valid_acc': valid_out['accuracy'],
            'valid_f1': valid_out['f1_weighted'],
            'test_loss': test_out['loss'],
            'test_acc': test_out['accuracy'],
            'test_f1': test_out['f1_weighted'],
        }
        history.append(row)

        print(
            f"Epoch {epoch}: train_loss={train_loss:.4f} | "
            f"valid_f1={valid_out['f1_weighted']:.4f} | test_f1={test_out['f1_weighted']:.4f}"
        )

        if valid_out['f1_weighted'] > best_valid_f1:
            best_valid_f1 = valid_out['f1_weighted']
            model.save_pretrained(best_ckpt_dir)
            tokenizer.save_pretrained(best_ckpt_dir)
            torch.save({'best_valid_f1': best_valid_f1, 'epoch': epoch}, run_dir / 'best_meta.pt')
            print('Saved best checkpoint:', best_ckpt_dir)

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / 'history.csv', index=False)

    # Reload best checkpoint for final reporting
    best_model = AutoModelForSequenceClassification.from_pretrained(best_ckpt_dir).to(device)
    final_valid = run_eval(best_model, valid_loader, criterion)
    final_test = run_eval(best_model, test_loader, criterion)

    return {
        'run_dir': run_dir,
        'best_checkpoint': best_ckpt_dir,
        'tokenizer': tokenizer,
        'history': history_df,
        'valid_eval': final_valid,
        'test_eval': final_test,
    }

# 3) Fine-Tune DistilRoBERTa (`distilroberta-base`)

In [ ]:
DISTILROBERTA_MODEL_NAME = 'distilroberta-base'

distil_results = finetune_model(
    pretrained_name=DISTILROBERTA_MODEL_NAME,
    run_name='distilroberta_meld',
    train_df=train_df,
    valid_df=valid_df,
    test_df=test_df,
    epochs=3,
    batch_size=16,
    lr=2e-5,
    max_length=128,
)

distil_results['history']

In [ ]:
print('Best DistilRoBERTa checkpoint:', distil_results['best_checkpoint'])
print_eval_report(distil_results['valid_eval'], split_name='DistilRoBERTa valid')
print_eval_report(distil_results['test_eval'], split_name='DistilRoBERTa test')

plot_conf_mat(
    distil_results['test_eval']['labels'],
    distil_results['test_eval']['preds'],
    title='DistilRoBERTa Test Confusion Matrix',
)

In [ ]:
def predict_sentences(checkpoint_dir, sentences, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir).to(device)
    model.eval()

    enc = tokenizer(
        sentences,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors='pt',
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=1)

    rows = []
    for sent, pred, prob in zip(sentences, preds, probs):
        rows.append(
            {
                'text': sent,
                'pred_label_id': int(pred),
                'pred_label_name': label2name[int(pred)],
                'confidence': float(np.max(prob)),
            }
        )
    return pd.DataFrame(rows)

sample_sentences = [
    "I am really happy with this result!",
    "I am disappointed and sad about what happened.",
    "Wait, what? I did not expect this.",
    "This makes me so angry right now.",
    "I'm so mad at you",
    "Oh my god, I can't believe this is happening!"
]

predict_sentences(distil_results['best_checkpoint'], sample_sentences)

# 4) Regenerate MELD Text Features From a Fine-Tuned Checkpoint

This step keeps the original multimodal pickle structure and only replaces `videoText` with new text embeddings extracted from the fine-tuned model.

You can choose either checkpoint as feature source.

In [ ]:
FEATURE_SOURCE = 'distilroberta'

feature_ckpt = distil_results['best_checkpoint']

print('Using checkpoint for feature extraction:', feature_ckpt)

In [ ]:
def encode_texts_to_cls_embeddings(checkpoint_dir, texts, batch_size=64, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir).to(device)
    model.eval()

    all_embeds = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding utterances'):
        batch_texts = [str(t) for t in texts[i:i + batch_size]]
        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt',
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc, output_hidden_states=True)
            # CLS token embedding from final hidden layer.
            cls_embed = out.hidden_states[-1][:, 0, :].detach().cpu().numpy().astype(np.float32)
            all_embeds.append(cls_embed)

    return np.concatenate(all_embeds, axis=0)

# Load original pickle again to preserve all non-text modalities.
with open(DATA_PATH, 'rb') as f:
    original_data = list(pickle.load(f))

videoSentence_local = original_data[9]
new_video_text = {}

for vid, sent_list in tqdm(videoSentence_local.items(), desc='Building dialog embeddings'):
    embeds = encode_texts_to_cls_embeddings(feature_ckpt, sent_list, batch_size=64, max_length=128)
    new_video_text[vid] = embeds

original_data[3] = new_video_text

feature_tag = FEATURE_SOURCE.lower()
out_pkl = DATA_PATH.parent / f'meld_multimodal_features_{feature_tag}_ft.pkl'
with open(out_pkl, 'wb') as f:
    pickle.dump(original_data, f)

print('Saved updated MELD features to:', out_pkl)
print('Example text feature shape:', next(iter(new_video_text.values())).shape)